In [1]:
import numpy as np
from stl import mesh as stl_mesh
import plotly.graph_objects as go  # for visualization
import os
import trimesh


import gyroid_utils
from gyroid_utils.utils import reload_all
#reload_all()

working_path = os.getcwd()
print("Current working directory:", working_path)

[gyroid_utils] version 3.1.4 loaded
Current working directory: c:\Users\cofo\Documents\02 - GitHub\GYROIDS\notebooks


In [2]:
# size of domain
pz2 = 300
py2 = 300
px2 = 300

# --- Discretization of the domain ---
# Resolution in each axis, calculated as a funcion of the size of the gyroid's unit cell
dx_grid = px2 / 400
dy_grid = py2 / 400
dz_grid = pz2 / 400

# 1D coordinate arrays. np.arange(stop + step, step) includes the endpoint like MATLAB's colon with step.
x1 = np.arange(0, px2 + dx_grid, dx_grid)       # x positions from 0 to pz2 + dx_grid, and the step size is dx_grid
y1 = np.arange(0, py2 + dy_grid, dy_grid)       # y positions from 0 to py2, step dy_grid
z1 = np.arange(0, pz2 + dz_grid, dz_grid)       # z positions from 0 to pz2, step dz_grid

# Create 3D coordinate grids. indexing='ij' -> (X,Y,Z) follow x1,y1,z1 order like MATLAB.
x, y, z = np.meshgrid(x1, y1, z1, indexing='ij')
print(np.min(x),np.max(x))
print(np.min(y),np.max(y))
print(np.min(z),np.max(z))

print(f"x-axis resolution {np.size(x1)=}, y-axis resolution {np.size(y1)=}, z-axis resolution {np.size(z1)=}")
print(f"In total, {np.size(x)} voxels in the 3D grid")

0.0 300.0
0.0 300.0
0.0 300.0
x-axis resolution np.size(x1)=401, y-axis resolution np.size(y1)=401, z-axis resolution np.size(z1)=401
In total, 64481201 voxels in the 3D grid


In [ ]:
# --------- Create gyroids -----------
file_name = "gyroid-test"

# ---  Define Period of gyroid unit cell -------
# px: linear gradient along x + sinusoidal ripple driven by y
px = 70 + 60 * (x / px2) + 20 * np.sin(4 * np.pi * y / py2)
# py: linear gradient along y + sinusoidal ripple driven by z
py = 70 + 60 * (y / py2) + 20 * np.sin(4 * np.pi * z / pz2)
# pz: radial bull's-eye pattern — largest period at the centre of the xy-plane
r_xy = np.sqrt((x - px2 / 2)**2 + (y - py2 / 2)**2)
pz = 60 + 80 * (1 - r_xy / r_xy.max())


# ------- check for errors -------
#in the period of the gyroid is too small for the grid resolution, it will cause errors in the marching cubes algorithm
if np.min(pz) <= 5 * dz_grid :
    print(f"max pz: {np.max(pz)}, min pz: {np.min(pz)}, resultion dz_grid: {dz_grid}")
    raise ValueError("Period too small for the grid resolution. z-axis.")   
elif np.min(py) <= 5 * dy_grid :
    print(f"max pz: {np.max(py)}, min pz: {np.min(py)}, resultion dz_grid: {dy_grid}")
    raise ValueError("Period too small for the grid resolution. y-axis.")   
elif np.min(px) <= 5 * dx_grid:
    print(f"max pz: {np.max(px)}, min pz: {np.min(px)}, resultion dz_grid: {dx_grid}")
    raise ValueError("Period too small for the grid resolution. z-axis.")

# ------ Define Thickness function t ---   #can be from -1.4265847744427516 + 1.4265847744427516
t = np.zeros_like(x) + 0.5


# --- Gyroid scalar field v (isosurface at v=0 gives the gyroid surface) ---
v = np.abs( np.sin((2*np.pi/px)*x) * np.cos((2*np.pi/py)*y)
    + np.sin((2*np.pi/py)*y) * np.cos((2*np.pi/pz)*z)
    + np.sin((2*np.pi/pz)*z) * np.cos((2*np.pi/px)*x) ) - t # add thickness field
v=-v

# --- Visualize result ---

gyroid_utils.viz.twod_view_of_matrix(v, x1, y1, z1, 0, 0.01)

In [ ]:
reload_all()
from gyroid_utils import logger, set_log_level
set_log_level("INFO")

solid=  np.copy(v)
solid[solid > 0] = 1
solid[solid <= 0] = 0

test_matrice = gyroid_utils.voxel_overhang_tools.detect_overhangs(solid, x1, y1, z1, angle = 45, bridge=30)

#invert x and z axis to match the orientation of the gyroid in the 3D view
rotated_matrix = np.swapaxes(test_matrice, 1, 2)

gyroid_utils.viz.twod_view_of_matrix(rotated_matrix, x1, y1, z1, 0, 3)

In [ ]:
reload_all()
from gyroid_utils import logger, set_log_level
set_log_level("INFO")

best_print_matrix, new_x, new_y, new_z = gyroid_utils.voxel_overhang_tools.find_optimal_orientation(solid, x1, y1, z1, n=20, overhang_angle = 65, bridge_size=30, grid_sample_factor=0.9)

rotated_matrix = np.swapaxes(best_print_matrix, 1, 2)

gyroid_utils.viz.twod_view_of_matrix(rotated_matrix, new_x, new_z, new_y, 0, 3)

In [ ]:
reload_all()
set_log_level("DEBUG")
vertex, faces = gyroid_utils.mesh_tools.mesh_from_matrix(matrix = rotated_matrix,
                                         iso_level = 0.1,
                                         algo_step_size=1,
                                         x=new_x,
                                         y=new_z,
                                         z=new_y,)

vertex, faces = gyroid_utils.mesh_tools.simplify_mesh(vertex, faces, target=1000000)
#vertex, faces = gyroid_utils.mesh_tools.smooth_mesh(vertex, faces, smoothing_factor=0.35, iterations=15)

gyroid_utils.viz.save_mesh_as_html(faces = faces,
                      verts = vertex,
                      file_name="C:\\Users\\cofo\\Desktop\\gyroids generation\\nop",
                      save = True)